In [11]:
import pandas as pd
from datetime import datetime, timedelta, timezone
import time
import os
import MetaTrader5 as mt5

In [12]:
mt5.initialize()

login = os.environ["FTMO_DEMO_LOGIN"]
password = os.environ["FTMO_DEMO_PASSWORD"]
server = os.environ["FTMO_DEMO_SERVER"]

mt5.login(login=login, password=password, server=server)

False

In [13]:
ticks = mt5.copy_rates_from_pos("EURUSD", mt5.TIMEFRAME_H1, 0, 24*5)
df = pd.DataFrame(ticks)
df['datetime'] = pd.to_datetime(df['time'], unit='s')
df.set_index('datetime', inplace=True)
df

,time,open,high,low,close,tick_volume,spread,real_volume
datetime,,,,,,,,
2024-12-18 18:00:00,1734544800,1.04738,1.04808,1.04670,1.04690,5308,2,0
2024-12-18 19:00:00,1734548400,1.04690,1.04800,1.04688,1.04753,3899,2,0
2024-12-18 20:00:00,1734552000,1.04753,1.04923,1.04736,1.04863,3586,2,0
2024-12-18 21:00:00,1734555600,1.04863,1.04863,1.03733,1.03772,22635,2,0
2024-12-18 22:00:00,1734559200,1.03772,1.03804,1.03439,1.03650,15693,2,0
...,...,...,...,...,...,...,...,...
2024-12-26 13:00:00,1735218000,1.04018,1.04021,1.03992,1.04012,848,2,0
2024-12-26 14:00:00,1735221600,1.04012,1.04013,1.03926,1.03995,1859,2,0
2024-12-26 15:00:00,1735225200,1.03996,1.04057,1.03970,1.04026,2575,2,0


In [44]:
symbol = "EURUSD"

sessions = [
    ('asia_pacific', '00:00', '11:00', 'Etc/GMT-2', '13826810'),  # Combined Tokyo and Sydney session
    ('london', '10:00', '19:00', 'Etc/GMT-2', '14675921'),
    ('new_york', '15:00', '00:00', 'Etc/GMT-2', '13294079')
]


In [53]:
sessions_df_list = []
for name, start_time, end_time, timezone, color in sessions:
    session_df = df.between_time(start_time, end_time)
    if name == "new_york":
        print(session_df.index)
    session_df_summary = session_df.resample('D').agg(
        min=('low', 'min'),
        max=('high', 'max'),
        #mean=('bid', 'mean'),
        #median=('bid', 'median'),
        start_time=('time', lambda x: x.index.min()),
        end_time=('time', lambda x: x.index.max())
    )
    session_df_summary.sort_values(by="start_time", inplace=True)
    session_df_summary["name"] = name
    session_df_summary["color"] = color
    sessions_df_list.append(session_df_summary)

sessions_df = pd.concat(sessions_df_list).sort_values(by='start_time').dropna()
sessions_df[sessions_df.name=="new_york"]

DatetimeIndex(['2024-12-18 18:00:00', '2024-12-18 19:00:00',
               '2024-12-18 20:00:00', '2024-12-18 21:00:00',
               '2024-12-18 22:00:00', '2024-12-18 23:00:00',
               '2024-12-19 00:00:00', '2024-12-19 15:00:00',
               '2024-12-19 16:00:00', '2024-12-19 17:00:00',
               '2024-12-19 18:00:00', '2024-12-19 19:00:00',
               '2024-12-19 20:00:00', '2024-12-19 21:00:00',
               '2024-12-19 22:00:00', '2024-12-19 23:00:00',
               '2024-12-20 00:00:00', '2024-12-20 15:00:00',
               '2024-12-20 16:00:00', '2024-12-20 17:00:00',
               '2024-12-20 18:00:00', '2024-12-20 19:00:00',
               '2024-12-20 20:00:00', '2024-12-20 21:00:00',
               '2024-12-20 22:00:00', '2024-12-20 23:00:00',
               '2024-12-23 00:00:00', '2024-12-23 15:00:00',
               '2024-12-23 16:00:00', '2024-12-23 17:00:00',
               '2024-12-23 18:00:00', '2024-12-23 19:00:00',
               '2024-12-

,min,max,start_time,end_time,name,color
datetime,,,,,,
2024-12-18,1.03439,1.04923,2024-12-18 18:00:00,2024-12-18 23:00:00,new_york,13294079
2024-12-19,1.03472,1.04140,2024-12-19 00:00:00,2024-12-19 23:00:00,new_york,13294079
2024-12-20,1.03612,1.04474,2024-12-20 00:00:00,2024-12-20 23:00:00,new_york,13294079
2024-12-23,1.03842,1.04448,2024-12-23 00:00:00,2024-12-23 23:00:00,new_york,13294079
2024-12-24,1.03835,1.04100,2024-12-24 00:00:00,2024-12-24 23:00:00,new_york,13294079
2024-12-26,1.03951,1.04094,2024-12-26 00:00:00,2024-12-26 17:00:00,new_york,13294079


In [31]:
objects = [
    dict(
    type="20",
    name=f'{symbol} {i.date()} {row["name"]} session',
    color=f"{row['color']}",
    background="1",
    filling="1",
    date1=f"{int(row['start_time'].replace(tzinfo=None).timestamp())}",
    date2=f"{int(row['end_time'].replace(tzinfo=None).timestamp())}",
    value1=f"{row['min']}",
    value2=f"{row['max']}"
    )

    for i, row in sessions_df.iterrows()
]

obj_strings = []
for obj in objects:
    s = "<object>\n"
    for key, value in obj.items():
        s += f"{key}={value}\n"
    s += "</object>\n"
    obj_strings.append(s)

print(obj_strings[0])

<object>
type=20
name=EURUSD 2024-12-18 london session
color=14675921
background=1
filling=1
date1=1734544800
date2=1734548400
value1=1.0467
value2=1.04808
</object>



In [39]:
# Read the file content (replace 'file_path' with your file path)
file_path = r"C:\Users\vynde\AppData\Roaming\MetaQuotes\Terminal\49CDDEAA95A409ED22BD2287BB67CB9C\MQL5\Profiles\Templates\default.tpl"

with open(file_path, "r", encoding="utf-16") as file:
    content = file.read()

missing_strings = [obj_string for obj_string in obj_strings if obj_string not in content]
print(len(missing_strings), "of", len(obj_strings))

ending_tag="</window>\n</chart>"
position = content.find(ending_tag)
if position == -1:
    print(f"Ending tag '{ending_tag}' not found in the file.")
else:
    print(position)

17 of 17
30789


In [40]:
# Insert missing strings before the ending tag
updated_content = content[:position] + "\n" + "\n".join(missing_strings) + "\n" + content[position:]

# Write the updated content back to the file
with open(file_path, "w", encoding="utf-16") as file:
    file.write(updated_content)